In [78]:

import numpy as np
import pandas as pd
import os
import pickle
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor, Pool
from sklearn.preprocessing import LabelEncoder

# Cấu hình đường dẫn
MODEL_DIR = "../models"
DATA_DIR = "../data"
os.makedirs(MODEL_DIR, exist_ok=True)

# ===== 1. LOGIC DOMAIN (Giữ nguyên logic hay của bạn) =====
TYPE_RELATIONS = {
    "Coffee Shop": ["Cafe", "Tea Shop", "Bakery"],
    "Restaurant": ["Bar/Pub", "Vegetarian", "Bakery"],
    "Bar/Pub": ["Restaurant"],
    "Natural": ["Attraction", "Entertainment"],
    "Attraction": ["Natural", "Cultural", "Historical"],
    "Cultural": ["Attraction", "Historical"],
    "Historical": ["Attraction", "Cultural"],
    "Hotel": ["Villa", "Resort", "Homestay", "Hostel", "Apartment"],
    "Shopping": ["Entertainment"],
    "Entertainment": ["Shopping", "Natural"],
}
TYPE_RELATIONS_LOWER = {k.lower(): [v.lower() for v in vals] for k, vals in TYPE_RELATIONS.items()}

def calculate_similarity_score(user_type, poi_type):
    """Tạo Feature mới: Độ phù hợp giữa sở thích và địa điểm"""
    u = str(user_type).lower().strip()
    p = str(poi_type).lower().strip()
    
    if u == p: 
        return 1.2
    
    # Check relation
    related_types = TYPE_RELATIONS_LOWER.get(u, [])
    if p in related_types:
        return 1.05
        
    return 0.9

# ===== 2. CLASS RECOMMENDER ENGINE HOÀN CHỈNH =====
class TravelRecommender:
    def __init__(self):
        self.model = None
        self.encoders = {} # Lưu encoder để dùng lại khi predict
        self.cat_features = ["user_city", "user_type", "poi_city", "poi_type"]
        
    def _feature_engineering(self, df):
        """Tạo thêm cột dữ liệu thông minh"""
        df = df.copy()
        
        # 1. Tính điểm tương đồng Type (Feature quan trọng nhất)
        df['type_match_score'] = df.apply(
            lambda x: calculate_similarity_score(x['user_type'], x['poi_type']), 
            axis=1
        )
        
        # 2. Xử lý dữ liệu thiếu
        cols_to_fill = ['user_price', 'price', 'rating']
        for col in cols_to_fill:
            if col in df.columns:
                df[col] = df[col].fillna(0)
                
        return df

    def train(self, data_path):
        print("Loading data...")
        df = pd.read_csv(data_path)
        
        # Tiền xử lý & Tạo feature
        df = self._feature_engineering(df)
        
        # Xử lý Label Encoding cho các cột Category (Để lưu lại quy tắc mapping)
        for col in self.cat_features:
            le = LabelEncoder()
            # Chuyển về string để tránh lỗi mixed types
            df[col] = df[col].astype(str)
            df[col] = le.fit_transform(df[col])
            self.encoders[col] = le # Lưu encoder lại!

        # Định nghĩa X và y
        # Thêm 'type_match_score' vào features
        features = self.cat_features + ["user_price", "price", "rating", "latitude", "longitude", "type_match_score"]
        X = df[features]
        y = df['label']

        # Chia tập train/test
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        print(f"Training CatBoost with features: {features}...")
        
        # Cấu hình CatBoost (Model mạnh nhất cho tabular data)
        self.model = CatBoostRegressor(
            iterations=500,
            learning_rate=0.05,
            depth=6,
            loss_function='RMSE',
            eval_metric='MAE',
            verbose=100,
            allow_writing_files=False
        )

        # Train model
        self.model.fit(
            X_train, y_train,
            eval_set=(X_test, y_test),
            # Catboost xử lý category cực tốt nếu ta chỉ định index của nó
            # Nhưng ở đây ta đã LabelEncode rồi nên coi như numerical cũng được
        )
        
        print("Model Score (R2):", self.model.score(X_test, y_test))
        self.save_model()

    def save_model(self):
        """Lưu cả Model và Encoders"""
        payload = {
            "model": self.model,
            "encoders": self.encoders
        }
        path = os.path.join(MODEL_DIR, "travel_recommender.pkl")
        with open(path, "wb") as f:
            pickle.dump(payload, f)
        print(f"✓ Model saved to {path}")

    def load_model(self):
        path = os.path.join(MODEL_DIR, "travel_recommender.pkl")
        with open(path, "rb") as f:
            payload = pickle.load(f)
            self.model = payload["model"]
            self.encoders = payload["encoders"]
        print("✓ Model loaded successfully")

    def predict(self, user_profile, list_poi_df):
        """
        Dự đoán cho 1 user với danh sách POI
        user_profile: dict {'user_city': '...', 'user_type': '...', 'user_price': ...}
        list_poi_df: DataFrame chứa danh sách POI
        """
        # 1. Tạo DataFrame kết hợp User + POI (Cartesian Product logic)
        predict_df = list_poi_df.copy()
        predict_df['user_city'] = user_profile['user_city']
        predict_df['user_type'] = user_profile['user_type']
        predict_df['user_price'] = user_profile['user_price']
        
        # Đổi tên cột cho khớp với lúc train (poi_id, name... giữ nguyên để hiển thị)
        # Giả sử list_poi_df có cột 'city_norm' -> 'poi_city', 'type' -> 'poi_type'
        predict_df.rename(columns={'city_norm': 'poi_city', 'type': 'poi_type', 'price_level': 'price'}, inplace=True)

        # 2. Feature Engineering (Tính điểm type match)
        predict_df = self._feature_engineering(predict_df)

        # 3. Encode dữ liệu (Dùng encoder đã train)
        for col in self.cat_features:
            le = self.encoders[col]
            # Xử lý trường hợp nhãn mới không có trong lúc train (Unseen labels)
            predict_df[col] = predict_df[col].astype(str).map(
                lambda s: le.transform([s])[0] if s in le.classes_ else -1
            )

        # 4. Chọn features để predict
        features = self.cat_features + ["user_price", "price", "rating", "latitude", "longitude", "type_match_score"]
        X_pred = predict_df[features]

        # 5. Predict
        predict_df['score'] = self.model.predict(X_pred)
        
        # 6. Trả về kết quả đã sort
        return predict_df.sort_values(by='score', ascending=False)


engine = TravelRecommender()
engine.train("../data/recommender_training.csv")
    


Loading data...
Training CatBoost with features: ['user_city', 'user_type', 'poi_city', 'poi_type', 'user_price', 'price', 'rating', 'latitude', 'longitude', 'type_match_score']...
0:	learn: 0.0819622	test: 0.0823532	best: 0.0823532 (0)	total: 12.5ms	remaining: 6.23s
100:	learn: 0.0121889	test: 0.0121785	best: 0.0121785 (100)	total: 524ms	remaining: 2.07s
200:	learn: 0.0050394	test: 0.0050735	best: 0.0050735 (200)	total: 1s	remaining: 1.49s
300:	learn: 0.0027231	test: 0.0027527	best: 0.0027527 (300)	total: 1.46s	remaining: 968ms
400:	learn: 0.0017132	test: 0.0017318	best: 0.0017318 (400)	total: 1.92s	remaining: 473ms
499:	learn: 0.0012209	test: 0.0012375	best: 0.0012375 (499)	total: 2.38s	remaining: 0us

bestTest = 0.001237496736
bestIteration = 499

Model Score (R2): 0.9995800720105606
✓ Model saved to ../models\travel_recommender.pkl


In [79]:
# # Test Predict thử (Giả lập)
# engine.load_model()
# sample_pois = pd.read_csv("../data/POI.csv").head(10) # Giả sử đã clean
# user = {'user_city': 'da nang', 'user_type': 'natural', 'user_price': 1}
# results = engine.predict(user, sample_pois)
# print(results[['name', 'score']])

In [80]:
import pandas as pd
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

class RouteOptimizer:
    def __init__(self, duration_file, poi_file):
        print("Loading Data...")
        # Fillna = 0 và ép kiểu an toàn
        self.duration_matrix = pd.read_csv(duration_file, index_col='poi_id').fillna(0)
        self.poi_df = pd.read_csv(poi_file).set_index('poi_id')
        print("✓ Data Loaded.")

    def get_time_between(self, from_id, to_id):
        """Lấy thời gian di chuyển, ép kiểu int"""
        try:
            val = self.duration_matrix.loc[from_id, to_id]
            return int(val) 
        except KeyError:
            return 100000 

    def optimize_route(self, selected_poi_ids, start_poi_id, max_time_minutes=720, visit_time_per_poi=60):
        # --- 1. CHUẨN BỊ DỮ LIỆU ---
        # Loại bỏ start_poi_id khỏi danh sách điểm đến để tránh lặp
        targets = [pid for pid in selected_poi_ids if pid != start_poi_id]
        
        # Danh sách node: [Depot (Hotel), Point1, Point2, ...]
        full_route_ids = [start_poi_id] + targets
        idx_to_id = {i: pid for i, pid in enumerate(full_route_ids)}
        n_locations = len(full_route_ids)

        # Build Matrix (Dictionary lồng nhau an toàn cho C++)
        time_matrix = {}
        for i in range(n_locations):
            time_matrix[i] = {}
            for j in range(n_locations):
                if i == j:
                    time_matrix[i][j] = 0
                else:
                    travel = self.get_time_between(idx_to_id[i], idx_to_id[j])
                    # Nếu j là điểm đến (không phải quay về hotel), cộng thêm giờ chơi
                    visit = 0 if j == 0 else int(visit_time_per_poi)
                    time_matrix[i][j] = travel + visit

        # --- 2. CẤU HÌNH SOLVER ---
        manager = pywrapcp.RoutingIndexManager(n_locations, 1, 0)
        routing = pywrapcp.RoutingModel(manager)

        # Callback function
        def time_callback(from_index, to_index):
            from_node = manager.IndexToNode(from_index)
            to_node = manager.IndexToNode(to_index)
            return time_matrix[from_node][to_node]

        transit_callback_index = routing.RegisterTransitCallback(time_callback)
        routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

        # Ràng buộc thời gian (Dimension)
        routing.AddDimension(
            transit_callback_index,
            30,  # Slack
            int(max_time_minutes), # Capacity
            True, # start_cumul_to_zero
            "Time"
        )

        # Penalty: Cho phép bỏ điểm nếu quá xa/hết giờ
        penalty = 100000
        for node in range(1, n_locations):
            routing.AddDisjunction([manager.NodeToIndex(node)], penalty)

        # --- 3. CHIẾN LƯỢC TÌM KIẾM ---
        search_parameters = pywrapcp.DefaultRoutingSearchParameters()
        
        # Bản 9.9 vẫn cần chiến lược khởi tạo
        search_parameters.first_solution_strategy = (
            routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
        
        search_parameters.local_search_metaheuristic = (
            routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
        
        search_parameters.time_limit.seconds = 2

        # --- 4. GIẢI ---
        solution = routing.SolveWithParameters(search_parameters)

        if solution:
            route = []
            index = routing.Start(0)
            total_time = 0
            
            while not routing.IsEnd(index):
                node_idx = manager.IndexToNode(index)
                poi_id = idx_to_id[node_idx]
                
                # Lấy tên hiển thị
                name = poi_id
                if poi_id in self.poi_df.index:
                    name = self.poi_df.loc[poi_id, 'name']

                route.append({
                    "order": len(route) + 1,
                    "poi_id": poi_id,
                    "name": name,
                    "type": "Start" if len(route)==0 else "Visit"
                })
                
                index = solution.Value(routing.NextVar(index))
                
                if not routing.IsEnd(index):
                    next_node = manager.IndexToNode(index)
                    total_time += time_matrix[node_idx][next_node]

            return {
                "status": "success",
                "total_time_min": total_time,
                "route": route
            }
        else:
            return {"status": "fail", "msg": "Không tìm được lộ trình hợp lệ."}

In [81]:

    # Test data giả lập (thay bằng file thật của bạn)
import json
    
# Giả sử bạn đã load file thật, ở đây tôi tạo dummy data để code chạy được ngay
optimizer = RouteOptimizer(
    "../data/duration_min.csv", 
    "../data/POI.csv"
)

sample_hotel = "hotel01815" 
sample_pois = ["attraction00001", "attraction00005", "restaurant00973"]
    
    # Chạy thử
res = optimizer.optimize_route(sample_pois, sample_hotel, max_time_minutes=600)
print(json.dumps(res, indent=2, ensure_ascii=False))

Loading Data...
✓ Data Loaded.
{
  "status": "success",
  "total_time_min": 233,
  "route": [
    {
      "order": 1,
      "poi_id": "hotel01815",
      "name": "Hue Riverside Boutique Resort & Spa",
      "type": "Start"
    },
    {
      "order": 2,
      "poi_id": "attraction00005",
      "name": "The Marble Mountains",
      "type": "Visit"
    },
    {
      "order": 3,
      "poi_id": "attraction00001",
      "name": "My Khe Beach",
      "type": "Visit"
    }
  ]
}


In [82]:

from sklearn.cluster import KMeans

class MultiDayPlanner:
    def __init__(self, recommender, optimizer, poi_df):
        self.recommender = recommender
        self.optimizer = optimizer
        self.poi_df = poi_df

    def plan_itinerary(self, user_profile, total_days, start_poi_id, spots_per_day=8):
        """
        Tạo lịch trình nhiều ngày.
        :param user_profile: dict thông tin user
        :param total_days: số ngày đi (VD: 3)
        :param start_poi_id: ID khách sạn (VD: 'hotel01815')
        :param spots_per_day: trung bình số điểm tham quan 1 ngày
        """
        
        # --- BƯỚC 1: LẤY GỢI Ý TỪ MODEL CATBOOST ---
        # Lấy dư ra (1.5 lần) để có đủ điểm cho việc gom nhóm
        num_pois_needed = total_days * spots_per_day
        buffer_limit = int(num_pois_needed * 1.5)
        
        print(f"1. Đang tìm {buffer_limit} địa điểm phù hợp nhất...")
        
        # Gọi hàm predict (trả về DataFrame đã sort theo score)
        ranked_pois_df = self.recommender.predict(user_profile, self.poi_df)
        
        # Lấy Top N điểm cao nhất
        top_pois = ranked_pois_df.head(buffer_limit).copy()
        
        # Loại bỏ khách sạn khỏi danh sách điểm đến (nếu lỡ có)
        top_pois = top_pois[top_pois['poi_id'] != start_poi_id]
        
        print(f"-> Đã chọn được {len(top_pois)} điểm tiềm năng.")

        # --- BƯỚC 2: GOM NHÓM ĐỊA LÝ (K-MEANS CLUSTERING) ---
        # Gom các điểm gần nhau vào cùng 1 ngày để tránh đi lại zig-zag
        print("2. Đang phân chia khu vực (Clustering)...")
        
        coords = top_pois[['latitude', 'longitude']].values
        
        # Nếu số điểm ít hơn số ngày -> Giảm số cụm K lại
        k_clusters = min(total_days, len(top_pois))
        
        if k_clusters > 0:
            kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
            top_pois['day_cluster'] = kmeans.fit_predict(coords)
        else:
            top_pois['day_cluster'] = 0

        # 

        # --- BƯỚC 3: TỐI ƯU HÓA LỘ TRÌNH TỪNG NGÀY (OR-TOOLS) ---
        full_itinerary = []

        actual_day = 1  # ✅ Đếm số ngày thực sự được tạo

        unique_clusters = sorted(top_pois['day_cluster'].unique())

        for cluster_id in unique_clusters:
            if actual_day > total_days:
                break

            print(f"--- Đang tối ưu hóa Ngày {actual_day} (Khu vực {cluster_id}) ---")

            # Lấy danh sách POI trong cluster
            day_pois_ids = top_pois[top_pois['day_cluster'] == cluster_id]['poi_id'].tolist()

            if not day_pois_ids:
                continue

            optimization_result = self.optimizer.optimize_route(
                selected_poi_ids=day_pois_ids,
                start_poi_id=start_poi_id,
                max_time_minutes=720
            )

            if optimization_result['status'] == 'success' or optimization_result['status'] == 'Optimal':
                # Chỉ giữ những POI thực sự được đi
                visited_pois = [
                    step for step in optimization_result['route']
                    if step['type'] == 'Visit'
                ]

                if len(visited_pois) == 0:
                    continue

                full_itinerary.append({
                    "day": actual_day,
                    "date": f"Ngày {actual_day}",
                    "total_time_min": optimization_result.get('total_time_min', 0),
                    "poi_count": len(visited_pois),
                    "route": optimization_result['route']
                })

                actual_day += 1
            else:
                print(f"⚠️ Không tối ưu được ngày {actual_day}: {optimization_result.get('msg', '')}")
        
        return full_itinerary

In [ ]:

# 1. Load các module con
rec_engine = TravelRecommender()
rec_engine.load_model() # Nhớ train trước khi load
    
# Load data POI đầy đủ để lấy tọa độ
df_poi = pd.read_csv("../data/POI.csv") # Cần clean data trước giống lúc train
# (Lưu ý: Bạn cần dùng hàm clean data giống hệt file RecommenderEngine cũ)
# Ví dụ: df_poi = rec_engine._feature_engineering(df_poi) ... 
    
opt_engine = RouteOptimizer(
    "../data/duration_min.csv",
    "../data/POI.csv"
)

    # 2. Khởi tạo Planner
planner = MultiDayPlanner(rec_engine, opt_engine, df_poi)

    # 3. User request: Đi 5 ngày (Test 5 ngày cho nhanh, 10 ngày tương tự)
user_req = {
    'user_city': 'da lat', 
    'user_type': 'natural', 
    'user_price': 1
}

final_itinerary = planner.plan_itinerary(
        user_profile=user_req, 
        total_days=3,
        start_poi_id='hotel01964',
        spots_per_day=4 # Mỗi ngày đi khoảng 4 điểm
    )
    
print(json.dumps(final_itinerary, indent=2, ensure_ascii=False))

✓ Model loaded successfully


TypeError: RouteOptimizer.__init__() takes 3 positional arguments but 4 were given